# Importing Libraries

In this section, we import the main libraries required for data analysis and modeling:

- **numpy** (`np`) — numerical operations and array handling.  
- **pandas** (`pd`) — working with tabular data (DataFrames).  
- **matplotlib.pyplot** (`plt`) and **seaborn** (`sns`) — data visualization and plotting.  
- **sklearn.model_selection.train_test_split** — splitting the dataset into training and testing sets.  
- **sklearn.metrics** (`classification_report`, `confusion_matrix`) — evaluating classification performance.  

Additionally:  
- **os** is imported to interact with the file system (e.g., listing files in `/kaggle/input`).  
- **warnings** is used to suppress unnecessary warning messages during execution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Loading the Dataset

We load the training, testing, and label data from the provided CSV files:  

- `train.csv` — training features  
- `test.csv` — testing features  
- `trainLabels.csv` — target labels for training  

Since the datasets do not contain column names, we manually assign them:  
- Feature columns are named `col_1`, `col_2`, … up to the total number of features.  
- The target column is added as `"label"` to train table.  

Finally, we display the data to verify its structure and use `.describe().T` to get a statistical summary of all features.

In [ ]:
train=pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
test=pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
train_labels=pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)
train

In [ ]:
train_labels

In [ ]:
train.columns=['col_'+str(i) for i in range(1,train.shape[1]+1)]
test.columns=['col_'+str(i) for i in range(1,train.shape[1]+1)]
train['label']=train_labels
train

In [ ]:
train.describe().T

# Data Analysis 

## Correlation check

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(train.corr(),cmap='coolwarm')

The heatmap above shows the **correlation matrix** of all features in the dataset.  
It helps to identify linear relationships between pairs of variables:

- Strong positive correlation → values close to **+1** (red).  
- Strong negative correlation → values close to **-1** (blue).  
- Weak or no correlation → values close to **0** (light colors).

### Key Observations
- `col_13` and `col_15` show noticeable correlations with several other features.  
  - This may indicate redundancy or potential multicollinearity.  
  - Such features might need to be checked during feature selection.  
- Most other features have low correlations (close to 0), suggesting relative independence.  
- The diagonal line of red squares represents each feature’s perfect correlation with itself.

## Count plot

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.countplot(data=train,x='label')

The bar chart shows the count of the target variable (`label`):

### Key Observations
- **Class 0**: slightly less frequent.  
- **Class 1**: appears a bit more often.  

Although class **1** is more common than class **0**, the difference is small.  
This means there is **no significant class imbalance**, and standard classification models can be trained without applying heavy resampling or class-weight adjustments.

## Violinplot

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.violinplot(data=train,x='label',y='col_13')

The violin plot shows the distribution of **feature `col_13`** across the two target classes (`label = 0` and `label = 1`):

### Key Observations
- For **class 0**, values of `col_13` are centered around negative values, with a wider spread below zero.  
- For **class 1**, the distribution shifts upward, with more values concentrated around positive numbers.  

This indicates that `col_13` has **good discriminative power** and may help separate the two classes.

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.violinplot(data=train,x='label',y='col_15')

The violin plot shows the distribution of **feature `col_15`** across both classes:

### Key Observations

- For **class 0**, the feature values are mostly centered below zero.  
- For **class 1**, the distribution shifts slightly higher, with more density around values above zero.  

Similar to `col_13`, the feature `col_15` also shows a **clear difference between classes**, making it a potentially useful predictor for classification.

## Scatterplot

Scatterplots help visualize the relationship between features and how they separate across classes (`label = 0` and `label = 1`). Each point represents one sample, colored by its class.

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.scatterplot(data=train, x='col_13',y='col_29', hue='label')

### `col_13` vs `col_29`
- The scatterplot shows a mild linear trend between `col_13` and `col_29`.  
- Both classes overlap significantly, though `class 1` tends to dominate higher ranges.  
Useful in combination with other features, but not strongly discriminative on its own.

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.scatterplot(data=train, x='col_13',y='col_15', hue='label')

### `col_13` vs `col_15`
- Clear separation between classes:  
  - **Class 0** tends to cluster in the lower-left region.  
  - **Class 1** is more spread toward positive values of both features.  
This pair of features provides good discriminative power.

In [ ]:
plt.figure(figsize=(10,8))
plt.grid(True)
sns.scatterplot(data=train, x='col_13',y='col_23', hue='label')

### `col_13` vs `col_23`
- The classes overlap more compared to the previous plot.  
- There is still a slight shift, but not as strong as with `col_15`.  
col_23` may add some complementary information but is weaker for classification.

# Random Forest Classifier

We use a **Random Forest** model to classify the data. Random Forest is an ensemble method that builds multiple decision trees and combines their predictions to improve accuracy and reduce overfitting.

### Parameters used:
- **n_estimators** → the number of trees in the forest.  
  - More trees usually improve performance but increase computation time.  
  - We test different values (from 50 to 500 in steps of 50) to find a good balance.  

- **max_depth = 5** → the maximum depth of each decision tree.  
  - Restricting depth prevents trees from growing too complex and overfitting.  
  - A shallow depth (like 5) forces trees to focus on the most important splits.  

- **random_state = 42** → ensures reproducibility of results.  
  - Fixing the seed makes experiments consistent across runs.  

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X=train.drop('label',axis=1)
y=train['label']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.25, random_state=42)

In [ ]:
errors = list()
for i in range(50,501,50):
    clf = RandomForestClassifier(n_estimators=i, max_depth=5, random_state=42)
    clf.fit(X_train,y_train)
    pred=clf.predict(X_test)
    errors.append(np.mean(pred!=y_test))

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(range(50,501,50),errors, '-o')
plt.grid(True)
plt.ylabel('Error')
plt.xlabel('n_estimators')

In [ ]:
clf = RandomForestClassifier(n_estimators=250, max_depth=5, random_state=42)
clf.fit(X_train,y_train)
pred=clf.predict(X_test)

In [ ]:
print(confusion_matrix(y_test,pred),'\n',classification_report(y_test,pred))

Out of 130 samples from class **0**, the model correctly predicted **109** and misclassified **21**.  
Out of 120 samples from class **1**, the model correctly predicted **107** and misclassified **13**.  

- **Precision (0.89 for class 0, 0.84 for class 1)** the model makes relatively few false positives.  
- **Recall (0.84 for class 0, 0.89 for class 1)** the model successfully captures most true positives for both classes.  
- **F1-scores (0.87 and 0.86)** balanced tradeoff between precision and recall.  

### Overall Performance
- **Accuracy = 0.86 (86%)** the model correctly predicts 86% of all cases.  
- **Macro and Weighted Averages ≈ 0.86** the performance is balanced across both classes, confirming there is no major bias toward one class. 

# K-Nearest Neighbors (KNN)

KNN is a simple, instance-based classification algorithm.  
Instead of learning explicit model parameters, it predicts the class of a new sample based on the majority class of its **k nearest neighbors** in the training set.

### Parameters used:
- **n_neighbors (k)** → the number of neighbors to consider.  
  - Small `k` (e.g., 1–3): model is highly flexible but prone to noise (overfitting).  
  - Large `k`: predictions are smoother but may ignore important local structure (underfitting).  
  - We test values of `k` from **1 to 49** to find the optimal balance.  

- **fit(X_train, y_train)** → stores the training data, since KNN is a *lazy learner* (no explicit training phase).  
- **predict(X_test)** → assigns labels to test samples by majority voting among the `k` closest neighbors (distance-based, usually Euclidean).  

### Model Selection
- We track the **error rate** for each tested value of `k`.  
- The error is plotted against the number of neighbors, helping to identify the best choice of `k`.  
- In this case, the chosen value is **n_neighbors = 6**, which achieves good performance without overfitting.

### Evaluation
- After selecting `k = 6`, the model is retrained and tested.  
- The **confusion matrix** and **classification report** provide final performance metrics.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
errors=[]
for i in range(1,50):
    knn=KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train)
    pred=knn.predict(X_test)
    errors.append(np.mean(pred!=y_test))

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(range(1,50),errors, '-o')
plt.grid(True)
plt.ylabel('Error')
plt.xlabel('Number of neighbors')

In [ ]:
knn=KNeighborsClassifier(n_neighbors=6)
knn.fit(X_train,y_train)
pred=knn.predict(X_test)
print(confusion_matrix(y_test,pred),'\n',classification_report(y_test,pred))

Out of 130 samples from class **0**, the model correctly classified **119** and misclassified **11**.  
Out of 120 samples from class **1**, the model correctly classified **109** and misclassified **11**.  

- **Class 0** precision = 0.92, recall = 0.92, F1 = 0.92  
- **Class 1** precision = 0.91, recall = 0.91, F1 = 0.91  
- **Overall accuracy = 0.91 (91%)**  
- Macro and weighted averages = 0.91, showing **balanced performance** across both classes.  

### Interpretation
- The KNN model with **k = 6** performs very well, achieving **91% accuracy** with nearly identical metrics for both classes.  
- The confusion matrix confirms that **misclassifications are minimal and evenly distributed**.  
- Compared to the Random Forest (accuracy ≈ 86%), KNN provides **better performance** on this dataset, suggesting the data has a local structure that KNN captures effectively.  

### Conclusion
KNN with `k = 6` is a strong candidate model, with **high accuracy and balanced precision/recall** across both classes.

In [ ]:
pred=knn.predict(test)

submission = pd.DataFrame({
    "Id": range(1, len(pred)+1),
    "Solution": pred
})

submission.to_csv("submission.csv", index=False)